In [ ]:
DROP = False
%run "/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/code/Initialization/Init/Initialization.ipynb"
start_date = "2020-06-01"

In [ ]:
# df = df_sp500
df = df.loc["2020-05-31":"2022-06-01"]
df.index = pd.to_datetime(df.index)
df.drop(columns=["ticker"],inplace=True)
if start_date:
    df = df[df.index >= start_date]

In [ ]:
df = df["VIX"]

In [ ]:
predictions_df = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/Predictions/predictions_gru.pkl")
directions_df = predictions_df -  predictions_df.shift()
ModelName = "GRU"

In [ ]:
df = pd.merge(directions_df,df, left_index=True, right_index=True)

In [ ]:
df = df.rename(columns={"predictions_gru":"predicted_value"})

In [ ]:
import requests
import pandas as pd
import numpy as np
from math import floor
from termcolor import colored as cl
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (30, 18)
plt.style.use('fivethirtyeight')

def get_vix(price, slow, fast, smooth):
    exp1 = price.ewm(span = fast, adjust = False).mean()
    exp2 = price.ewm(span = slow, adjust = False).mean()
    vix = pd.DataFrame(exp1 - exp2).rename(columns = {'VIX':'vix'})
    signal = pd.DataFrame(vix.ewm(span = smooth, adjust = False).mean()).rename(columns = {'vix':'signal'})
    hist = pd.DataFrame(vix['vix'] - signal['signal']).rename(columns = {0:'hist'})
    frames =  [vix, signal, hist]
    df = pd.concat(frames, join = 'inner', axis = 1)
    return df

df_vix = get_vix(df['VIX'], 26, 12, 9)
df_vix

def implement_vix_strategy(prices, data):    
    buy_price = []
    sell_price = []
    vix_signal = []
    no_operation = []
    no_operation_buy = []
    no_operation_sell = []
    signal = -1

    for i in range(len(data)):
        if data['predicted_value'][i] > 0:
            if signal != 1:
                buy_price.append(prices[i])
                sell_price.append(np.nan)
                signal = 1
                vix_signal.append(signal)
                no_operation.append(np.nan)
                no_operation_buy.append(np.nan)
                no_operation_sell.append(np.nan)
            else:
                buy_price.append(np.nan)
                sell_price.append(np.nan)
                vix_signal.append(0)
                no_operation.append(np.nan)
                no_operation_buy.append(prices[i])
                no_operation_sell.append(np.nan)
        elif data['predicted_value'][i] < 0:
            if signal != -1:
                buy_price.append(np.nan)
                sell_price.append(prices[i])
                signal = -1
                vix_signal.append(signal)
                no_operation.append(np.nan)
                no_operation_buy.append(np.nan)
                no_operation_sell.append(np.nan)
            else:
                buy_price.append(np.nan)
                sell_price.append(np.nan)
                vix_signal.append(0)
                no_operation.append(np.nan)
                no_operation_buy.append(np.nan)
                no_operation_sell.append(prices[i])
        else:
            buy_price.append(np.nan)
            sell_price.append(np.nan)
            vix_signal.append(0)
            no_operation.append(prices[i])
            no_operation_buy.append(np.nan)
            no_operation_sell.append(np.nan)
    return buy_price, sell_price, vix_signal,no_operation,no_operation_buy,no_operation_sell
          

In [ ]:
def plot_vix(prices, hist):
    ax1 = plt.subplot2grid((11,1), (0,0), rowspan = 5, colspan = 1)
    ax2 = plt.subplot2grid((11,1), (5,0), rowspan = 3, colspan = 1)
    ax3 = plt.subplot2grid((11,1), (8,0), rowspan = 3, colspan = 1)

    ax1.plot(prices)

    for i in range(len(prices)):
        if str(hist[i])[0] == '-':
            ax2.bar(prices.index[i], hist[i], color = '#ef5350')
        else:
            ax2.bar(prices.index[i], hist[i], color = '#26a69a')

    plt.legend(loc = 'lower right')

# plot_vix(df['VIX'], df_vix['vix'], df_vix['signal'], df_vix['hist'])

In [ ]:
def get_benchmark(start_date, investment_value):
    vix = df['VIX']#get_historical_data('SPY', start_date)['VIX']
    benchmark = pd.DataFrame(data=np.diff(vix))
    benchmark = benchmark.rename(columns={0:"benchmark_returns"})#np.diff(spy))

    investment_value = investment_value
    number_of_stocks = floor(investment_value/vix[0])
    benchmark_investment_ret = []
    for i in range(len(benchmark['benchmark_returns'])):
        returns = number_of_stocks*benchmark['benchmark_returns'][i]
        benchmark_investment_ret.append(returns)

    benchmark_investment_ret_df = pd.DataFrame(benchmark_investment_ret).rename(columns = {0:'investment_returns'})
    return benchmark_investment_ret_df

In [ ]:
# %matplotlib inline

buy_price, sell_price, vix_signal, no_operation,no_operation_buy,no_operation_sell = implement_vix_strategy(df['VIX'], df)
ax1 = plt.subplot2grid((11,1), (0,0), rowspan = 5, colspan = 1)
ax2 = plt.subplot2grid((11,1), (5,0), rowspan = 3, colspan = 1)
ax3 = plt.subplot2grid((11,1), (8,0), rowspan = 3, colspan = 1)

ax1.plot(df['VIX'], color = 'skyblue', linewidth = 2, label = ModelName+' VIX')
ax1.plot(df.index, buy_price, marker = '^', color = 'green', markersize = 10, label = 'BUY OPERATION', linewidth = 0)
ax1.plot(df.index, sell_price, marker = 'v', color = 'r', markersize = 10, label = 'SELL OPERATION', linewidth = 0)
ax1.plot(df.index, no_operation, marker = '+',fillstyle='none', color = 'blue', markersize = 10, label = 'HOLD', linewidth = 0)
ax1.plot(df.index, no_operation_buy, marker = '^',fillstyle='none', color = 'green', markersize = 10, label = 'BUY DECISION (Not Executed)', linewidth = 0)
ax1.plot(df.index, no_operation_sell, marker = 'v',fillstyle='none', color = 'r', markersize = 10, label = 'SELL DECISION (Not Executed)', linewidth = 0)
# print("fdsafdsa")
# print(ax1.get_legend_handles_labels.)
ax1.legend()
ax1.set_title(ModelName+' Direction Forecast on VIX Index\nPress-Release',size = 50)
ax1.set_ylabel("Investment Operations",fontsize = 30)
ax2.set_ylabel("Portfolio Value ",fontsize = 30)

ax3.set_ylabel("Revenue",fontsize = 30)

position = []
for i in range(len(vix_signal)):
    if vix_signal[i] > 1:
        position.append(0)
    else:
        position.append(1)
        
for i in range(len(df['VIX'])):
    if vix_signal[i] == 1:
        position[i] = 1
    elif vix_signal[i] == -1:
        position[i] = 0
    else:
        position[i] = position[i-1]

vix = df_vix['vix']
signal = df_vix['signal']
VIX_price = df['VIX']
vix_signal = pd.DataFrame(vix_signal).rename(columns = {0:'vix_signal'}).set_index(df.index)
position = pd.DataFrame(position).rename(columns = {0:'vix_position'}).set_index(df.index)

frames = [VIX_price, vix, signal, vix_signal, position]
strategy = pd.concat(frames, join = 'inner', axis = 1)



df_ret = pd.DataFrame(np.diff(df['VIX'])).rename(columns = {0:'returns'})
vix_strategy_ret = []


for i in range(len(df_ret)):
    try:
        returns = df_ret['returns'][i]*strategy['vix_position'][i]
        vix_strategy_ret.append(returns)
    except:
        pass
    
vix_strategy_ret_df = pd.DataFrame(vix_strategy_ret).rename(columns = {0:'vix_returns'})

investment_value = 100000
number_of_stocks = floor(investment_value/df['VIX'][0])
print
vix_investment_ret = []

for i in range(len(vix_strategy_ret_df['vix_returns'])):
    returns = number_of_stocks*vix_strategy_ret_df['vix_returns'][i]
    vix_investment_ret.append(returns)



investment_ret_df = pd.DataFrame(vix_investment_ret).rename(columns = {0:'investment_returns'})
total_investment_ret = round(sum(investment_ret_df['investment_returns']), 2)
profit_percentage = floor((total_investment_ret/investment_value)*100)
print(cl("Profit gained from the "+ModelName +" strategy by investing $100k in df : {}".format(total_investment_ret), attrs = ['bold']))
print(cl("Profit percentage of the "+ModelName +" strategy : {}%".format(profit_percentage), attrs = ['bold']))


investment_ret_df.index = vix_signal[1:].index
investment_cumsum = (investment_ret_df.cumsum()+100000)
investment_cumsum.index = vix_signal[1:].index
# ax2.plot(investment_ret_df, color = 'gray', linewidth = 5, label = 'sell OPERATION')
#####################################################################################################
ax2.plot(investment_cumsum, color = 'gray', linewidth = 2, label = 'Strategy Cumulative Total')
#####################################################################################################

benchmark= get_benchmark('2020-06-01', 100000)
benchmark_cum = (benchmark.cumsum()+100000)
benchmark_cum.index = df[1:].index
benchmark.index= df[1:].index
#####################################################################################################
ax2.plot(benchmark_cum, color = 'skyblue', linewidth = 2, label = 'Benchmark Cumulative Total')#label = ModelName+'VIX')
plt.rcParams['font.size'] = 20
#####################################################################################################

hist = pd.DataFrame(investment_ret_df).rename(columns = {0:'hist'})
hist.index = vix_signal[1:].index
print(hist)
for i in range(len(hist)):
    if str(hist['investment_returns'][i])[0] == '-':
        ax3.bar(hist.index[i], hist['investment_returns'][i], color = '#ef5350')
    else:
        ax3.bar(hist.index[i], hist['investment_returns'][i], color = '#26a69a')
ax3.plot(investment_ret_df, color = 'gray', linewidth = 2, label = 'Strategy Operation Revenue')#label = ModelName+'VIX')

investment_value = 100000
total_benchmark_investment_ret = round(sum(benchmark['investment_returns']), 2)
benchmark_profit_percentage = floor((total_benchmark_investment_ret/investment_value)*100)

print(cl('Benchmark profit by investing $100k : {}'.format(total_benchmark_investment_ret), attrs = ['bold']))
print(cl('Benchmark Profit percentage : {}%'.format(benchmark_profit_percentage), attrs = ['bold']))
print(cl(ModelName +' Strategy profit is {}% higher than the Benchmark Profit'.format(profit_percentage - benchmark_profit_percentage), attrs = ['bold']))

plt.legend(loc='best')
plt.show()